# MLIP Active-Learning Tutorial

> New to ALF? Start with the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **offline (pool-based) active-learning loop**. Our use case: starting from
a pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximise a fitness*. Here we do the opposite kind of active
learning: we **minimise model error using as few expensive labels as possible**. Each label is an
expensive quantum-chemistry calculation (DFT). We work from a fixed pool of DFT-labelled aspirin
configurations and let the model decide which ones are worth "paying" to reveal — choosing the
configurations where its committee of models *disagrees most* (often the most informative ones,
though, as we'll see, not always).

### Experiment overview

1. Download a pretrained MACE organics model and a pool of DFT-labelled aspirin configurations.
2. Finetune a committee (ensemble) of models on a small seed set and use their disagreement to
   estimate prediction uncertainty.
3. Each round: score the remaining candidate pool, acquire the most uncertain configurations,
   reveal their DFT labels, and finetune again.
4. Compare an uncertainty-driven acquisition against a random baseline, and see why acquisition
   choice matters.

### Framework Components

1. **Dataset** (`AspirinDataset`, defined below): loads DFT-labelled aspirin configurations and
   splits them into seed-train / validation / test / **candidate pool**. The pool holds the
   unlabelled configurations active learning chooses from.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): serves the remaining candidate pool each round.
4. **Acquisition Function** ([`UncertaintyBased`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/) from `alf_tools`): selects the configurations where the committee disagrees most (highest prediction variance). [`RandomSelection`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/) is the baseline for comparison.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) with the dataset as scorer): reveals the precomputed DFT energy (and forces) for an acquired configuration via `dataset.query`.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

## Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository.
The MLIP tutorial needs the `mlip` package (the MACE force field), which ALF exposes through the
optional `mlip` extra (mirrored as the `mlip` dependency group in `tutorials/pyproject.toml`). From
the `tutorials/` directory, sync that group so the extra is installed alongside the tutorial
dependencies:

```bash
uv sync --group mlip   # installs alf_core, alf_tools[mlip] and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

`uv sync` installs the **CPU** build of PyTorch by default. For GPU acceleration, see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to
install this tutorial's dependencies into the current kernel, then restart the kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install "alf_tools[mlip]" matplotlib pandas huggingface_hub

### Step 0: Download the Model and Dataset

The cell below downloads the pretrained MACE organics model and the public **rMD17 aspirin** dataset
(both from Hugging Face, public, no credentials) into the local cache. The aspirin configurations
carry DFT energies and forces, so no live quantum-chemistry engine is needed.

In [ ]:
from pathlib import Path

# Pretrained MACE organics foundation model -> alf models dir, so the MLIPModel loader
# finds it locally and skips its (private) S3 fallback.
import alf_tools.models.utils.mlip_utils as mlip_utils  # noqa: E402
from huggingface_hub import hf_hub_download, snapshot_download

models_dir = mlip_utils._MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)

# Public rMD17 aspirin dataset (DFT energies + forces) from the mlip tutorials collection.
DATA_DIR = Path("data/aspirin")
snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="training/rmd17_aspirin_*",
    local_dir=str(DATA_DIR),
)
ASPIRIN_TRAIN_XYZ = DATA_DIR / "training" / "rmd17_aspirin_train.xyz"

print(f"✅ Pretrained model present at {models_dir / 'mace_organics_02.zip'}")
print(f"✅ rMD17 aspirin pool present at {ASPIRIN_TRAIN_XYZ}")

### Step 1: Import Required Libraries

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    Candidate,
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from alf_tools.optimizer.acquisition_functions import UCB, RandomSelection
from ase.io import read as ase_read

print("✅ All imports successful!")

### Step 2: Load the rMD17 Aspirin Dataset (the molecule pool)

We load configurations of **aspirin** from the revised MD17 (rMD17) dataset — these are the pool
of candidate molecules we optimise over. Our objective is **stability**, defined as `-energy` (eV):
the most stable conformer is the lowest-energy one, and ALF maximises, so we store `-energy` as the
label. We subsample `N_POOL` configurations and let ALF split them into a small **seed-train** set,
a **validation** set (the surrogate needs it for finetuning), and a **candidate pool** to search.

In [ ]:
DATA_SEED = 51505
N_POOL = 150  # subsample of rMD17 aspirin configs (the molecule pool to optimise over)


def load_aspirin(xyz_path: Path | str, max_configs: int | None = None, seed: int = 0) -> LabelledCandidates:
    """Read aspirin configurations into `LabelledCandidates`.

    The optimisation objective is **stability**, defined as `-energy` (eV): higher stability
    means a lower-energy, more stable conformer. ALF maximises labels, so storing stability lets
    `UCB`, `get_top_k`, and the recall/regret metrics all treat "higher = better".
    """
    atoms_list = ase_read(str(xyz_path), index=":")
    if max_configs is not None and len(atoms_list) > max_configs:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(atoms_list), size=max_configs, replace=False)
        atoms_list = [atoms_list[i] for i in idx]
    candidates = [Candidate(data=atoms, modality=Modality.STRUCTURE) for atoms in atoms_list]
    stability = np.asarray([-atoms.get_potential_energy() for atoms in atoms_list])
    return LabelledCandidates(candidates=candidates, labels=stability)


class AspirinDataset(BaseDataset):
    """Pre-labelled aspirin conformers (label = stability = -energy), split into seed / val / pool."""

    def __init__(self, config: BaseDatasetConfig, data: LabelledCandidates):
        """Store the pre-built labelled data; `setup()` performs the split."""
        super().__init__(config)
        self._data = data

    def load_dataset(self) -> LabelledCandidates:
        """Return the pre-built `LabelledCandidates` (stability labels)."""
        return self._data


def make_aspirin_dataset(data: LabelledCandidates | None = None, seed: int = DATA_SEED) -> AspirinDataset:
    """Build and set up an `AspirinDataset` with the CPU-tiny split used in this tutorial."""
    if data is None:
        data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_POOL, seed=seed)
    ds = AspirinDataset(
        BaseDatasetConfig(
            name="rmd17_aspirin",
            modality=Modality.STRUCTURE,
            seed=seed,
            train_ratio=0.12,
            validation_frac=0.2,
            test_ratio=0.0,  # Bayesian optimisation tracks the best in the pool; no held-out test
            split_type="random",
            problem_type=ProblemType.REGRESSION,
        ),
        data,
    )
    ds.setup()
    return ds


data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_POOL, seed=DATA_SEED)

# Fail fast if aspirin contains any element the pretrained organics model never saw.
from mlip.models.mace.network import Mace  # noqa: E402

_pretrained_ff = mlip_utils._load_model_from_zip(Mace, "mace_organics_02.zip")
_ztable = set(_pretrained_ff.dataset_info.atomic_energies_map.keys())
_missing = {int(n) for c in data.candidates for n in c.data.numbers} - _ztable
if _missing:
    raise ValueError(f"aspirin contains elements outside the pretrained z-table: {_missing}")

dataset = make_aspirin_dataset(data)
print(dataset)
print(
    f"✅ rMD17 aspirin pool ready — seed-train={len(dataset.train_dataset)}, "
    f"pool={len(dataset.candidate_pool)} (objective: maximise stability = -energy)"
)

### Step 3: The Oracle (precomputed DFT)

The oracle is the expensive ground-truth evaluator. Here each aspirin configuration already has a
DFT energy, so the oracle simply **reveals** that label on demand: `Oracle(scorer=dataset)` calls
`dataset.query(...)` to look up the energy for an acquired configuration. This mirrors production
active learning, where labelling is costly and is therefore spent sparingly on the most informative
structures.

> **Offline vs online.** This is the **offline** setting: the oracle matches acquired candidates to
> their labels *by object identity* against the dataset, so it only works for configurations already
> in the pool. If instead you want to **generate new structures on the fly** and label them with a
> live evaluator (a running DFT/xTB calculation, or a pretrained MLIP used *as* the oracle), use a
> `BaseModel`-backed oracle and a generative search — see the
> [online design tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/online_design_tutorial.ipynb).

In [ ]:
# Offline oracle: reveal the precomputed DFT label for an acquired configuration.
# In production this is an expensive DFT calculation; here the labels already exist in
# the dataset, so the oracle is a lookup (`dataset.query`) keyed by candidate identity.
oracle = Oracle(scorer=dataset)
print("✅ Oracle ready (offline lookup of precomputed DFT labels)!")

### Step 4: Search and Acquisition

`DatasetSearch` serves the remaining **candidate pool**. `UCB` (Upper Confidence Bound) scores each
candidate by `μ + α·σ` — the committee's mean predicted stability plus `α` times its uncertainty —
so it balances **exploiting** molecules predicted to be very stable against **exploring** ones the
committee is unsure about. `RandomSelection` is the baseline that picks at random, to show the
surrogate-guided search actually finds the best molecule faster.

In [ ]:
# DatasetSearch serves the remaining candidate pool to the acquisition each round.
search_fn = DatasetSearch()
print("✅ Search ready (offline candidate pool)!")

### Step 5: The Surrogate — a Committee of Finetuned MACE Models

Our surrogate is an ensemble (committee) of `MLIPModel`s, each finetuning the **same** pretrained
MACE model on a different bootstrap resample of the labelled set, so they disagree where data is
scarce. `EnsembleWrapper.predict` returns the mean predicted **stability** and the variance across
members — both consumed by `UCB`.

*Note:* the surrogate predicts the stability label (`-energy`); MACE re-fits a per-element
reference energy from the training data. For a single molecule (constant composition) that fit is
degenerate but harmless — it subtracts the correct constant — so stability is reproduced on a
consistent scale.

In [ ]:
def mlip_factory(seed: int) -> MLIPModel:
    """Build a finetuning MLIPModel (from the pretrained MACE model) with the given seed."""
    return MLIPModel(
        model_config=MLIPModelConfig(model_path="mace_organics_02.zip"),
        train_config=MLIPTrainConfig(epochs=8, batch_size=2, learning_rate=1e-3),
        seed=seed,
    )


def make_surrogate(n_members: int = 2) -> Surrogate:
    """Build a committee surrogate of finetuned MACE models (1 member = no uncertainty)."""
    return Surrogate(
        model=EnsembleWrapper(
            model_factory=mlip_factory,
            config=EnsembleWrapperConfig(
                base_seed=0,
                n_members=n_members,
                subsample=SubsampleConfig(fraction=1.0, replace=True),
            ),
        )
    )


surrogate = make_surrogate(n_members=2)
print("✅ Surrogate committee initialised (2 finetuned MACE members)!")

### Step 6: Optimizer and Design Task

`UncertaintyBased` ranks candidates purely by the committee's prediction variance, so each round we
label the pool configurations the ensemble disagrees on most — classic query-by-committee active
learning. The `Optimizer` combines it with the `DatasetSearch` over the candidate pool; `DesignTask`
runs the rounds.

In [ ]:
acquisition_fn = UCB(alpha=1.0)  # mu + alpha*sigma over predicted stability; raise alpha to explore more
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

num_acq_rounds = 3
acq_batch_size = 3
task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)
print(f"✅ Optimizer + DesignTask ready ({num_acq_rounds} rounds × {acq_batch_size} labels)!")

### Step 7: Run the Active-Learning Experiment (uncertainty-driven)

Each round: take the remaining candidate pool, score it with the committee, acquire the most
uncertain configurations, reveal their DFT labels, finetune the committee, and evaluate on the
held-out test set.

> **⏱️ Slowest cell.** This is the most expensive step in the notebook: it finetunes a 2-member
> MACE committee once per acquisition round (plus the initial fit). On a laptop CPU expect a few
> minutes; it is much faster on GPU. The random baseline below is comparable in cost.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)  # keep MACE output quiet in the notebook

uncertainty_path = Path("results/mlip_design/")
if uncertainty_path.exists():
    shutil.rmtree(uncertainty_path)
loggers = [TerminalStateLogger(), FileStateLogger(output_path=uncertainty_path)]

state = task.setup(dataset=dataset, surrogate=surrogate)
print("🚀 Running uncertainty-driven active learning...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Uncertainty-driven experiment completed!")

### Step 8: Random-Sampling Baseline

To isolate the effect of the *acquisition strategy*, the baseline runs the identical loop on the
same aspirin splits with the **same 2-member committee surrogate** — the only thing that changes is
*how* configurations are chosen (uniformly at random instead of by committee variance). Keeping the
surrogate fixed makes the two learning curves a fair, apples-to-apples comparison.

> **⏱️ Slow cell** (comparable to Step 7): another committee finetuned across acquisition rounds.

In [ ]:
# Rebuild a fresh dataset so the baseline starts from the same initial splits.
dataset_random = make_aspirin_dataset()
random_oracle = Oracle(scorer=dataset_random)  # oracle is bound to THIS dataset's labels

# Same 2-member committee as the UncertaintyBased run — only the acquisition differs.
random_optimizer = Optimizer(acquisition_fn=RandomSelection(seed=0), search_fn=search_fn)
random_surrogate = make_surrogate(n_members=2)
random_task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

random_path = Path("results/mlip_design_random/")
if random_path.exists():
    shutil.rmtree(random_path)
random_loggers = [TerminalStateLogger(), FileStateLogger(output_path=random_path)]

random_state = random_task.setup(dataset=dataset_random, surrogate=random_surrogate)
print("🚀 Running random-selection baseline...")
random_task.run(
    random_state, state_loggers=random_loggers, optimizer=random_optimizer, oracle=random_oracle
)
print("✅ Baseline completed!")

### Step 9: Results

The headline metric is the **learning curve**: test-set energy error against the number of labels
acquired. Because both runs share the same committee surrogate, any difference is down to the
acquisition strategy alone. We then check absolute accuracy with **energy and force parity plots**
on the held-out test set — forces matter as much as energies for a usable force field, and MACE is
trained on both.

In [ ]:
uncertainty_metrics = pd.read_csv("results/mlip_design/metrics.csv")
rnd_metrics = pd.read_csv("results/mlip_design_random/metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLIP Active Learning: uncertainty-driven vs random", fontsize=15, fontweight="bold")

for df, label, color in [
    (uncertainty_metrics, "UncertaintyBased (committee)", "#e74c3c"),
    (rnd_metrics, "Random (committee)", "#3498db"),
]:
    axes[0].plot(
        df["dataset/num_train"], df["surrogate/test_mse"], marker="o", label=label, color=color
    )
    axes[1].plot(
        df["dataset/num_train"], df["surrogate/test_spearman"], marker="o", label=label, color=color
    )

axes[0].set_xlabel("Number of labelled structures")
axes[0].set_ylabel("Test energy MSE (eV²)")
axes[0].set_title("Learning curve (lower is better)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of labelled structures")
axes[1].set_ylabel("Test Spearman")
axes[1].set_title("Rank correlation (higher is better)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

**Reading the learning curves.** The left panel is the headline result: test-set energy MSE (eV²,
lower is better) on the held-out test *trajectory* against the number of labelled structures, which
grows by `acq_batch_size` each round. The right panel tracks rank correlation (Spearman, higher is
better) over the same budget. Both curves use the **same 2-member committee** — only the acquisition
differs — so any separation is attributable to the strategy, and the test set is a disjoint
trajectory so the numbers reflect genuine generalisation rather than leakage.

On a problem this small — 150 pool configs, a 2-member committee, and only a few rounds — expect the
two curves to be close and a little noisy: bootstrap resampling of a handful of structures gives
only a coarse uncertainty signal, so small gaps are not decisive. The point here is that the
end-to-end loop runs and the error trends downward as labels are spent; we return to *why*
uncertainty sampling need not win on a toy problem in the conclusion.

Next we check absolute accuracy with energy and force parity plots.

In [ ]:
# Energy and force parity on the held-out test trajectory, using the random-baseline committee.
test_cands = dataset_random.test_dataset.candidates
test_true = dataset_random.test_dataset.labels
test_pred = random_surrogate.predict(test_cands).means  # committee-mean energy

# The committee wrapper aggregates energies but not forces, so we read forces from one
# trained member (a finetuned MLIPModel) via its predict_with_forces API.
member = random_surrogate.model.members[0]
_, pred_forces = member.predict_with_forces(test_cands)
true_forces = [np.asarray(c.features["forces"]) for c in test_cands]
pred_f = np.concatenate([f.ravel() for f in pred_forces])
true_f = np.concatenate([f.ravel() for f in true_forces])

energy_mae = float(np.mean(np.abs(test_pred - test_true)))
force_mae = float(np.mean(np.abs(pred_f - true_f)))
force_rmse = float(np.sqrt(np.mean((pred_f - true_f) ** 2)))
print(f"Test energy MAE: {energy_mae:.4f} eV")
print(f"Test force  MAE: {force_mae:.4f} eV/Å   RMSE: {force_rmse:.4f} eV/Å")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].scatter(test_true, test_pred, alpha=0.7, color="#9b59b6")
elims = [min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())]
axes[0].plot(elims, elims, "k--", alpha=0.5)
axes[0].set_xlabel("DFT energy (eV)")
axes[0].set_ylabel("Predicted energy (eV)")
axes[0].set_title(f"Energy parity (MAE {energy_mae:.3f} eV)")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(true_f, pred_f, alpha=0.3, s=8, color="#16a085")
flims = [min(true_f.min(), pred_f.min()), max(true_f.max(), pred_f.max())]
axes[1].plot(flims, flims, "k--", alpha=0.5)
axes[1].set_xlabel("DFT force component (eV/Å)")
axes[1].set_ylabel("Predicted force component (eV/Å)")
axes[1].set_title(f"Force parity (MAE {force_mae:.3f} eV/Å)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Reading the parity plots.** Each point on the **left** is a held-out configuration: its DFT energy
(x) against the finetuned model's prediction (y), with the dashed line marking perfect agreement.
The **right** panel does the same for every force *component* (each atom contributes 3). Points
hugging the diagonal mean the model reproduces DFT faithfully. Energies span only a narrow window
(~1–2 eV about the mean), so a constant vertical offset there would signal a residual
reference-energy (E0) mismatch rather than poorly learned energy *differences*. Forces are the more
demanding test — they are the gradient the model must get right to be usable in a simulation, and
MACE is trained on them directly — so the force MAE (eV/Å) is the metric to watch as you scale the
experiment up.

### Step 10 (bonus): MLIP Embeddings for Downstream Tasks

A finetuned MLIP is also a learned **featuriser**: pooling its per-atom invariant features into one
fixed-length vector per structure gives an embedding you can reuse elsewhere — train a cheap
downstream regressor/classifier on it, or cluster configurations by structural and energetic
similarity.

A dedicated `MLIPModel.get_embeddings()` method is coming in a follow-up PR; the intended call is
shown (commented) below. For now we stand in **placeholder vectors** so the downstream pipeline —
*extract → reduce → visualise* — runs end-to-end and shows the shape of the workflow.

In [ ]:
# --- Intended API (coming in a follow-up PR; not yet implemented on MLIPModel) ---
# member = random_surrogate.model.members[0]
# embeddings = member.get_embeddings(test_cands)  # (n_structures, embed_dim) per-structure features

# Placeholder so this section runs today: synthetic stand-in embeddings.
# Swap in the commented call above once MLIPModel.get_embeddings() lands.
EMBED_DIM = 256  # illustrative width of MACE's pooled invariant features
_emb_rng = np.random.default_rng(DATA_SEED)
embeddings = _emb_rng.standard_normal((len(test_cands), EMBED_DIM))
print(f"Embeddings: {embeddings.shape}  (placeholder — see note above)")

In [ ]:
# Project the per-structure embeddings to 2-D with PCA (numpy SVD) and colour by DFT energy.
X = embeddings - embeddings.mean(axis=0)
_u, _s, Vt = np.linalg.svd(X, full_matrices=False)
proj = X @ Vt[:2].T

plt.figure(figsize=(6, 5))
sc = plt.scatter(proj[:, 0], proj[:, 1], c=test_true, cmap="viridis", alpha=0.8)
plt.colorbar(sc, label="DFT energy (eV)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("MLIP embeddings (PCA) — placeholder vectors")
plt.tight_layout()
plt.show()

**Reading the embedding plot.** Each point is a test configuration projected onto the first two
principal components of its embedding, coloured by DFT energy. Because these are **placeholder
(random) vectors**, the scatter is an unstructured blob with no energy gradient — that is expected
here. With *real* MACE embeddings from `get_embeddings()`, you would instead expect structurally and
energetically similar configurations to sit close together (a visible energy gradient, or distinct
clusters) — exactly the structure that makes the embeddings useful for clustering or as inputs to a
cheap downstream model.

## Conclusion

We built an offline (pool-based) active-learning loop for a machine-learned interatomic potential
with ALF: finetune a pretrained MACE committee on a few DFT-labelled aspirin configurations,
estimate uncertainty from committee disagreement, acquire more configurations from a candidate pool,
and repeat. Every component is a standard ALF abstraction — `AspirinDataset`, the dataset-lookup
`Oracle`, `DatasetSearch`, the `EnsembleWrapper` surrogate, `UncertaintyBased`/`RandomSelection`,
`Optimizer`, and `DesignTask` — and both acquisition strategies share the same committee surrogate
on a leakage-free (disjoint-trajectory) test set.

This example is deliberately tiny and fast, so treat the learning curves as illustrative rather
than conclusive. On a problem this small, uncertainty sampling does not reliably beat random
selection — a useful reminder that the acquisition strategy must be matched to the problem and
validated, not assumed. Query-by-committee active learning is the workhorse of *production* MLIP
training, where it selects from large, diverse pools of physically meaningful structures.

### Next steps

- **Scale up** so the uncertainty signal can show: more committee members, more acquisition rounds,
  a larger pool, and a larger held-out test set (rMD17 aspirin ships 1200/900/900 configs).
- **Try a stronger uncertainty estimate**: a deep-kernel / last-layer GP or a last-layer Laplace
  approximation on the MACE features gives calibrated epistemic variance (the latter is the
  NTK-flavoured, more principled cousin of an ensemble).
- **Go online**: generate candidates on the fly and label them with a live oracle (DFT/xTB, or a
  pretrained MLIP used *as* the oracle) — see the offline-vs-online note in Step 3.
- **Try a different molecule** or train from scratch (`model_path=None`) for chemistries with no
  compatible foundation model.

**Happy modelling!** ⚛️🔬✨

In [ ]:
for p in [Path("results/mlip_design/"), Path("results/mlip_design_random/")]:
    if p.exists():
        shutil.rmtree(p)
print("✅ Results directories cleaned up!")